# EDA orientée NLP du corpus de travail

Cette analyse ne cherche pas à décrire un jeu de données, elle cherche à **fixer les réglages
d'un entraînement Seq2Seq** et à savoir ce qu'ils coûtent. Chaque section pose une question dont
la réponse change une valeur de configuration ou une conclusion du rapport. Une analyse dont la
réponse ne changerait rien n'a pas sa place ici.

Le corpus est **CNN/DailyMail**, configuration `3.0.0`, celle de See et al. 2017 : l'article
complet d'un côté, les puces de highlights de l'autre, entités nommées en clair. La référence y
fait trois à quatre phrases, ce qui distingue la tâche du résumé extrême et fixe deux réglages
mesurés plus bas : le plafond de tokens cible et le budget de décodage.

Un point de lecture à poser d'emblée. Le ROUGE-L rapporté ici n'est pas le ROUGE-Lsum que la
littérature publie sur ce corpus : `rougeLsum` découpe la référence sur ses sauts de ligne et
apparie chaque phrase séparément, quand ROUGE-L exige une seule sous-séquence traversant toute la
paire. Sur une référence de trois à quatre phrases, la seconde mesure est nettement plus sévère.
Toutes les comparaisons de ce projet sont internes, même métrique et même jeu de test pour tous
les modèles ; aucune n'est à confronter à un chiffre publié.

Le « 100 % » désigne partout les 20 000 exemples du corpus de travail figé, jamais les
287 113 exemples d'entraînement de CNN/DailyMail complet.

## Ce que le notebook ne fait pas

Il ne recalcule rien que la chaîne de données ait déjà mesuré. Les distributions de longueur, les
comptes de troncature et les sommes de contrôle sont produits par `python -m src.data.build` et
écrits à côté du corpus. Les relire ici, plutôt que les refaire, est ce qui garantit que le
notebook et l'entraînement parlent du même corpus.

Il ne trace pas non plus de graphique décoratif. Les seules figures présentes sont celles dont la
forme, et pas seulement la moyenne, change une décision.

> Les sorties de ce notebook sont retirées à chaque commit par le hook `nbstripout`. Les chiffres
> qui comptent sont donc recopiés dans la documentation, page **Données**, où ils sont durables.

In [ ]:
from __future__ import annotations

import json
import sys
from collections import Counter
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt

from src.data.config import load_pipeline_config
from src.data.dataset import load_manifest, load_split, load_training_proportion
from src.data.tokenize import build_tokenizer, token_lengths
from src.data.validate import require_no_leakage, validate_split

CONFIG = load_pipeline_config(REPO_ROOT / "configs" / "data" / "cnn_dailymail.yaml")
PROCESSED = REPO_ROOT / "data" / "processed" / "cnn_dailymail"

plt.rcParams["figure.figsize"] = (9.0, 4.0)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.linestyle"] = ":"

SCRATCH_COLOUR = "#c1440e"
PRETRAINED_COLOUR = "#1f4e79"

## 1. Ce que le corpus déclare de lui-même

Avant de mesurer quoi que ce soit, on lit ce que la chaîne de préparation a écrit. Le manifeste
porte une somme de contrôle par split et par proportion : c'est ce qui permet d'affirmer que le
corpus analysé ici est celui qui a servi aux neuf entraînements, et pas une reconstruction
approchante.

In [ ]:
manifest = load_manifest(PROCESSED)

print(f"corpus            {manifest.name} (source : {manifest.source_dataset})")
print(f"version           {manifest.dataset_version}")
print(f"graine du tirage  {manifest.seed}")
print(f"tokenizer         {manifest.tokenizer}")
print()
for split, size in manifest.splits.items():
    print(f"{split:<12} {size:>6} exemples   sha256 {manifest.split_checksums[split][:16]}...")

## 2. Volume disponible

La première question est aussi la plus simple : combien d'exemples, et de quoi dispose-t-on pour
chaque proportion de l'ablation ?

Ce qui compte ici n'est pas seulement le total, c'est **l'emboîtement**. Le sous-ensemble à 10 %
doit être un préfixe de celui à 50 %, lui-même préfixe de celui à 100 %. Sans cette propriété, un
écart entre deux points de la courbe mélangerait l'effet de la taille et celui de la composition
de l'échantillon, et rien dans le résultat ne permettrait de séparer les deux.

In [ ]:
splits = {name: load_split(PROCESSED, name) for name in ("train", "validation", "test")}
proportions = {percentage: load_training_proportion(PROCESSED, percentage) for percentage in (10, 50, 100)}

for name, examples in splits.items():
    print(f"{name:<12} {len(examples):>6}")

print()
identifiers = {
    percentage: [example.example_id for example in examples]
    for percentage, examples in proportions.items()
}
for percentage, examples in proportions.items():
    print(f"{percentage:>4} % du corpus d'entrainement : {len(examples):>6} exemples")

print()
print("10 % est un prefixe de 50 %  :", identifiers[50][: len(identifiers[10])] == identifiers[10])
print("50 % est un prefixe de 100 % :", identifiers[100][: len(identifiers[50])] == identifiers[50])

## 3. Qualité : valeurs manquantes, doublons, fuite entre splits

Trois défauts se ressemblent sur un tableau de bord et n'ont pas du tout les mêmes conséquences.

Un champ vide casse l'entraînement ou, pire, apprend au modèle à produire du vide. Un doublon
intra-split biaise la pondération d'un exemple. Un document présent à la fois dans
l'entraînement et dans le test invalide la mesure entière : le modèle est alors interrogé sur ce
qu'il a mémorisé.

La chaîne de données applique déjà ces contrôles au moment de la construction et refuse d'écrire
un corpus sale. Les rejouer ici sert à montrer le résultat, pas à le découvrir.

In [ ]:
for name, examples in splits.items():
    report = validate_split(name, examples)
    print(report.describe())
    print(f"   propre : {report.is_clean}")

print()
try:
    require_no_leakage(splits["train"], splits["validation"], splits["test"])
    print("Aucun chevauchement entre les splits, ni par identifiant ni par document.")
except Exception as error:
    print("FUITE :", error)

**Ce que la mesure donne.** Aucun champ vide, aucun identifiant dupliqué, aucun chevauchement
entre les splits, aucun résumé plus long que son document. Treize documents apparaissent deux fois
dans l'entraînement.

Ces treize cas sont signalés et conservés. Les retirer serait un nettoyage silencieux du corpus de
référence, qui rendrait les scores incomparables avec la littérature. Treize doublons sur 20 000
pèsent 0,065 % de la pondération, et ce sont des dépêches authentiquement republiées, pas une
erreur de chargement.

## 4. Longueur des documents et des résumés

C'est la section qui fixe `max_source_tokens` et `max_target_tokens`. Trois unités coexistent et
elles ne disent pas la même chose :

- les **caractères** mesurent le texte brut,
- les **mots** donnent une intuition lisible,
- les **tokens** sont la seule unité que le modèle voit, et la seule qui coûte du calcul.

Un plafond se choisit en tokens. Les deux autres unités servent à comprendre ce que le plafond
coupe réellement.

In [ ]:
statistics = json.loads((PROCESSED / "statistics.json").read_text(encoding="utf-8"))

header = f"{'split':<12}{'unite':<14}{'moyenne':>10}{'mediane':>10}{'p95':>10}{'max':>10}"
print(header)
print("-" * len(header))
for split in ("train", "validation", "test"):
    words = statistics["characters_and_words"][split]
    tokens = statistics["tokens"][split]
    for label, block in (
        ("mots source", words["source_words"]),
        ("tokens source", tokens["source_tokens"]),
        ("mots cible", words["target_words"]),
        ("tokens cible", tokens["target_tokens"]),
    ):
        print(
            f"{split:<12}{label:<14}{block['mean']:>10.1f}{block['median']:>10.1f}"
            f"{block['p95']:>10.1f}{block['maximum']:>10}"
        )
    print()

### Ce que le plafond coupe

Les statistiques ci-dessus sont mesurées **sans troncature**. Le tableau suivant chiffre l'effet du
plafond retenu et de quatre alternatives, sur deux axes qui ne se confondent pas : la part des
documents touchés, et la part du texte réellement conservée.

Un document coupé n'est pas un document perdu. Ce qui compte est la proportion de contenu qui
atteint l'encodeur.

In [ ]:
tokenizer = build_tokenizer(CONFIG.tokenizer.hf_id)
source_lengths, target_lengths = token_lengths(splits["train"], tokenizer, CONFIG.tokenizer)

cap_source = CONFIG.tokenizer.max_source_tokens
cap_target = CONFIG.tokenizer.max_target_tokens
total_source_tokens = sum(source_lengths)

print(f"plafond retenu : {cap_source} tokens source, {cap_target} tokens cible")
print()
print(f"{'plafond':>9}{'documents coupes':>20}{'texte conserve':>18}{'cout encodeur':>16}")
print("-" * 63)
for cap in (256, 512, 768, 1024, 1536):
    cut = sum(1 for length in source_lengths if length > cap)
    kept = sum(min(length, cap) for length in source_lengths)
    relative_cost = (cap / cap_source) ** 2
    marker = "  <-- retenu" if cap == cap_source else ""
    print(
        f"{cap:>9}{cut / len(source_lengths):>19.1%}{kept / total_source_tokens:>18.1%}"
        f"{relative_cost:>15.1f}x{marker}"
    )

over_target = sum(1 for length in target_lengths if length > cap_target)
print()
print(f"cibles coupees a {cap_target} tokens : {over_target}/{len(target_lengths)} = {over_target / len(target_lengths):.2%}")

**Décision, et son prix.** Le plafond de 512 tokens coupe **85 % des articles** et ne laisse
entrer que **la moitié du texte source**. C'est la limite dominante du projet sur ce corpus, et
elle est assumée plutôt que masquée : l'attention coûte le carré de la longueur, donc passer à
1 024 tokens pour récupérer 32 points de texte multiplierait par quatre le coût de l'encodeur.
Sur une carte portable de 8 Go, cela transformerait une campagne de quelques heures en une
campagne d'une journée. C'est aussi le budget d'encodage de la littérature T5 sur ce corpus ; les
travaux qui vont à 1 024 le font sur d'autres modèles et d'autres machines.

Deux choses tempèrent le chiffre. D'abord, **les deux familles de modèles subissent exactement la
même troncature**, donc la comparaison reste équitable : ce qui est perdu est une part du plafond
atteignable, pas l'équité. Ensuite, l'article de presse est écrit en pyramide inversée, l'essentiel
d'abord, et les puces de highlights suivent cet ordre : couper la queue coûte moins que ce que
« la moitié du texte » laisse craindre.

Le plafond des cibles, lui, coupe 5,3 % des résumés, et il est posé sur le p95 mesuré à 130
tokens. Le plafond de 64 tokens qu'un corpus de résumé extrême autoriserait en couperait 60 % :
c'est le réglage qui devait changer en premier avec le corpus.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11.0, 4.0))

axes[0].hist([min(length, 2000) for length in source_lengths], bins=60, color=PRETRAINED_COLOUR)
axes[0].axvline(cap_source, color=SCRATCH_COLOUR, linestyle="--", linewidth=2, label=f"plafond {cap_source}")
axes[0].set_title("Documents : longueur en tokens")
axes[0].set_xlabel("tokens (ecretes a 2000 pour l'affichage)")
axes[0].set_ylabel("exemples")
axes[0].legend()

axes[1].hist(target_lengths, bins=60, color=PRETRAINED_COLOUR)
axes[1].axvline(cap_target, color=SCRATCH_COLOUR, linestyle="--", linewidth=2, label=f"plafond {cap_target}")
axes[1].set_title("Resumes : longueur en tokens")
axes[1].set_xlabel("tokens")
axes[1].legend()

figure.tight_layout()
plt.show()

Les deux distributions n'ont pas la même forme, et c'est le point de cette figure.

Celle des documents est asymétrique, avec une queue qui s'étend jusqu'à 4 449 tokens, mais son
corps est déjà bien au-dessus du plafond : la médiane est à 899 tokens pour une troncature à 512.
Ce n'est donc pas la queue que le plafond coupe, c'est le corps de la distribution. Celle des
résumés est resserrée autour de 70 tokens avec une queue jusqu'à 413.

Un modèle de résumé sur ce corpus doit donc encaisser une entrée longue et systématiquement
tronquée, et produire une sortie de plusieurs phrases dont la longueur varie du simple au triple.
C'est ce second point qui fixe le budget de décodage : un plafond calé sur la médiane laisserait
la moitié des références hors d'atteinte.

## 5. Ratio de compression

Le ratio résumé sur document dit quelle tâche le modèle doit apprendre. Un ratio de 0,5 est une
reformulation, un ratio de 0,1 est une réécriture qui jette neuf dixièmes de l'entrée.

In [ ]:
ratios = sorted(target / source for source, target in zip(source_lengths, target_lengths, strict=False) if source)

percentile = lambda fraction: ratios[int(fraction * (len(ratios) - 1))]
print(f"moyenne  {sum(ratios) / len(ratios):.4f}")
print(f"mediane  {percentile(0.50):.4f}")
print(f"p5       {percentile(0.05):.4f}")
print(f"p95      {percentile(0.95):.4f}")
print()
print(f"compression mediane : le resume fait {percentile(0.50) * 100:.1f} % du document")

**Ce que cela impose.** Le ratio médian de 9 % ressemble à celui d'un corpus de résumé
extrême, et c'est un faux ami : ici l'article est long et la référence l'est aussi, trois à quatre
phrases. Le ratio seul ne dit pas quelle tâche le modèle doit apprendre, et la mesure qui le dit
est le recouvrement entre le résumé et son article.

| Recouvrement résumé → article | Entraînement | Test |
| --- | --- | --- |
| Unigrammes | 84,5 % | 85,5 % |
| Bigrammes | 46,5 % | 48,6 % |
| Trigrammes | 27,4 % | 29,0 % |

Cinq mots sur six du résumé figurent déjà dans le document, et près d'un bigramme sur deux. Deux
conséquences suivent.

La première touche la métrique : **la mesure reportée est ROUGE-L, pas ROUGE-Lsum**. `rougeLsum`
découpe la référence sur ses sauts de ligne et apparie chaque phrase séparément, quand ROUGE-L
exige une seule sous-séquence traversant toute la paire. Sur une référence de trois à quatre
phrases, ROUGE-L est nettement plus sévère, et les chiffres publiés sur ce corpus sont des
ROUGE-Lsum. Aucun chiffre de ce projet ne s'y compare.

La seconde touche le modèle from scratch, et elle joue en sa faveur : à ce niveau d'extractivité,
sélectionner et recopier les bons fragments est une stratégie payante. Le modèle n'a pas besoin de
reformuler pour marquer des points, ce qui est exactement le contraire de ce qu'un corpus de résumé
extrême lui demanderait.

## 6. Vocabulaire : la section qui décide du modèle from scratch

C'est l'analyse la plus importante du notebook pour ce projet, et celle qu'une EDA généraliste
n'aurait pas faite.

Le modèle from scratch et la baseline pré-entraînée partagent le tokenizer `t5-small`. Pour la
baseline, c'est un choix neutre : ses embeddings sont déjà entraînés. Pour le modèle from scratch,
la table d'embedding est **le plus gros bloc de paramètres du modèle**, et chacune de ses lignes
doit être apprise à partir de rien, uniquement sur les occurrences présentes dans le corpus.

La question devient donc : combien de lignes de cette table le corpus permet-il réellement
d'apprendre ?

In [ ]:
encoded_sources = tokenizer(
    [CONFIG.tokenizer.source_prefix + example.source for example in splits["train"]],
    truncation=False,
)["input_ids"]
encoded_targets = tokenizer([example.target for example in splits["train"]], truncation=False)["input_ids"]

counts: Counter[int] = Counter()
for identifiers_ in encoded_sources:
    counts.update(identifiers_)
for identifiers_ in encoded_targets:
    counts.update(identifiers_)

vocabulary = len(tokenizer)
observed = len(counts)
occurrences = sum(counts.values())

print(f"entrees du tokenizer          {vocabulary:>10,}")
print(f"types observes                {observed:>10,}  ({observed / vocabulary:.1%})")
print(f"jamais observes               {vocabulary - observed:>10,}  ({1 - observed / vocabulary:.1%})")
print(f"occurrences totales           {occurrences:>10,}")
print()
print(f"hapax (1 occurrence)          {sum(1 for c in counts.values() if c == 1):>10,}")
print(f"rares (< 10 occurrences)      {sum(1 for c in counts.values() if c < 10):>10,}")

In [ ]:
ordered = counts.most_common()

print("Concentration des occurrences :")
cumulative = 0
targets = [0.5, 0.9, 0.95, 0.99]
index_of = {}
for index, (_, count) in enumerate(ordered, start=1):
    cumulative += count
    while targets and cumulative >= targets[0] * occurrences:
        index_of[targets[0]] = index
        targets.pop(0)

for fraction, index in index_of.items():
    print(f"  {fraction:>5.0%} des occurrences tiennent dans {index:>6,} types ({index / vocabulary:.1%} de la table)")

In [ ]:
d_model = 256
embedding_parameters = vocabulary * d_model
dead_parameters = (vocabulary - observed) * d_model
model_parameters = 15_591_424  # scratch_100, releve dans reports/results/scratch_100/run.json

print(f"table d'embedding      {vocabulary:>7,} x {d_model} = {embedding_parameters:>12,} parametres")
print(f"modele complet                          {model_parameters:>12,} parametres")
print(f"part de l'embedding                     {embedding_parameters / model_parameters:>12.1%}")
print()
print(f"lignes jamais mises a jour              {dead_parameters:>12,} parametres")
print(f"soit                                    {dead_parameters / model_parameters:>12.1%} du modele")

**Le résultat.** Sur 32 100 entrées du tokenizer, **24 480 apparaissent au moins une fois**
dans les 21,3 millions d'occurrences du corpus d'entraînement, soit 76 %. Les 7 620 restantes
correspondent à 1,95 million de paramètres, **12,5 % du modèle from scratch, qui ne reçoivent
jamais le moindre gradient**. Ils sont initialisés au hasard, sauvegardés dans chaque checkpoint,
et n'apprennent rien.

La concentration aggrave le constat : **50 % des occurrences tiennent dans 92 types**, et il faut
15 863 types pour couvrir 99 % du texte. Les 2 326 types vus moins de dix fois ont une ligne
d'embedding mise à jour trop peu souvent pour valoir mieux que du bruit.

C'est l'explication mécanique à garder pour l'ablation d'architecture : la table d'embedding pèse
53 % du modèle à 4 couches, donc ajouter des couches fait varier une minorité des paramètres,
pendant que la majorité reste sous-entraînée. **Le levier n'est pas la profondeur, c'est le
vocabulaire.** Un tokenizer réduit au corpus, de l'ordre de 8 000 entrées, libérerait environ
6 millions de paramètres, mais casserait le partage du tokenizer avec la baseline et donc
l'équité de la comparaison. C'est un compromis explicite, pas un oubli.

In [ ]:
figure, axes = plt.subplots(figsize=(9.0, 4.0))

frequencies = [count for _, count in ordered]
axes.loglog(range(1, len(frequencies) + 1), frequencies, color=SCRATCH_COLOUR, linewidth=1.5)
axes.axvline(observed, color="#555555", linestyle="--", label=f"{observed:,} types observes")
axes.axvline(index_of[0.9], color=PRETRAINED_COLOUR, linestyle=":", label=f"90 % des occurrences : {index_of[0.9]:,} types")
axes.set_title("Distribution des frequences de tokens (echelle log-log)")
axes.set_xlabel("rang du token")
axes.set_ylabel("occurrences")
axes.legend()

figure.tight_layout()
plt.show()

La droite quasi parfaite sur une échelle log-log est une loi de Zipf, ce qui est attendu. Ce qui
l'est moins, et qui compte ici, est **l'endroit où la courbe s'effondre** : au-delà du rang 22 000,
les fréquences tombent sous dix occurrences. Tout ce qui est à droite de ce point est une ligne
d'embedding que ce corpus ne peut pas apprendre.

## 7. Fragmentation en sous-mots

Le tokenizer `t5-small` est un SentencePiece : il ne produit jamais de token inconnu, il découpe
en morceaux. L'absence d'`<unk>` n'est donc pas une bonne nouvelle en soi, elle déplace le
problème. Le taux de fragmentation dit combien.

In [ ]:
source_words = sum(len(example.source.split()) for example in splits["train"])
source_pieces = sum(len(identifiers_) for identifiers_ in encoded_sources)
target_words = sum(len(example.target.split()) for example in splits["train"])
target_pieces = sum(len(identifiers_) for identifiers_ in encoded_targets)

print(f"documents : {source_pieces / source_words:.3f} tokens par mot")
print(f"resumes   : {target_pieces / target_words:.3f} tokens par mot")
print()
print(f"un plafond de {cap_source} tokens vaut donc environ {cap_source / (source_pieces / source_words):.0f} mots")

**1,42 token par mot** est un taux sain pour de l'anglais journalistique : la majorité des mots
courants tiennent en un seul morceau, la fragmentation se concentre sur les noms propres. Cela
signifie surtout que le plafond de 512 tokens correspond à environ **360 mots**, à comparer à la
médiane de 634 mots par article.

Le rapprochement de ces deux chiffres est le constat central de ce notebook : **le plafond tombe
sous la médiane**. Il ne rogne pas une queue de distribution, il coupe l'article médian avant sa
moitié. C'est ce qui distingue cette troncature de celle d'un corpus de dépêches courtes, où le
même plafond ne touchait qu'une minorité de documents.

## 8. Exemples extrêmes

Les deux bouts de la distribution ne sont pas des anomalies à retirer : ils disent ce que le
modèle doit encaisser dans le pire cas.

In [ ]:
pairs = sorted(zip(source_lengths, splits["train"], strict=False), key=lambda item: item[0])

for label, (length, example) in (("Le plus court", pairs[0]), ("Le plus long", pairs[-1])):
    print(f"--- {label} : {length} tokens, {len(example.source.split())} mots ---")
    print("document :", example.source[:300].replace(chr(10), " "), "...")
    print("resume   :", example.target)
    print()

Le document le plus long, 6 191 tokens, est une page de rumeurs de mercato : une liste, pas un
article. Son résumé de référence ne peut être déduit d'aucune de ses phrases. C'est le type
d'exemple pour lequel la troncature à 512 tokens ne dégrade rien, parce que rien d'exploitable
n'était présent au-delà non plus.

Le plus court, 43 tokens, est un fait divers de trois phrases, et son résumé en reprend la
substance. C'est le cas favorable, et c'est aussi celui où un modèle from scratch a une chance
réelle.

Ces deux exemples sont conservés dans le corpus. Les retirer relèverait la moyenne pour avoir
écarté ce qui est difficile.

## 9. Budget d'entraînement

Dernière question, celle qui dimensionne la campagne : combien de tokens chaque proportion de
l'ablation représente-t-elle réellement, une fois la troncature appliquée ?

In [ ]:
print(f"{'proportion':>12}{'exemples':>11}{'tokens source':>16}{'tokens cible':>15}")
print("-" * 54)
for percentage, examples in proportions.items():
    size = len(examples)
    source_budget = sum(min(length, cap_source) for length in source_lengths[:size])
    target_budget = sum(min(length, cap_target) for length in target_lengths[:size])
    print(f"{percentage:>11} %{size:>11,}{source_budget:>16,}{target_budget:>15,}")

**Ce que cela veut dire pour l'ablation.** Le point à 10 % ne représente que 984 000 tokens
source, une fois la troncature appliquée. À titre de repère, `t5-small` a été pré-entraîné sur de
l'ordre de 34 milliards de tokens, soit environ **35 000 fois plus**. Le modèle from scratch part
donc de zéro avec quatre ordres de grandeur de moins de texte que ce dont dispose déjà la baseline
avant même de commencer.

Ce chiffre, mesuré ici, est le meilleur repère pour lire la courbe du rapport : la comparaison
n'oppose pas deux modèles à données égales, et multiplier par dix un corpus de 20 000 exemples ne
change pas l'ordre de grandeur.

## Ce que l'EDA a fixé

| Constat | Décision |
| --- | --- |
| Médiane des articles à 899 tokens, coût quadratique de l'attention | `max_source_tokens = 512`, 85 % d'articles coupés, 50 % du texte gardé |
| p95 des résumés à 130 tokens, 60 % coupés à 64 | `max_target_tokens = 128`, 5,3 % de cibles coupées, `max_new_tokens` aligné |
| 85 % des articles au plafond, gaspillage de padding mesuré à 3,8 % | regroupement par longueur conservé, mais il ne rapporte plus que 2,8 points |
| 76 % du vocabulaire observé, 50 % des occurrences dans 92 types | le levier du modèle from scratch est le vocabulaire, pas la profondeur |
| 85 % d'unigrammes et 47 % de bigrammes du résumé déjà dans l'article | tâche extractive, ROUGE-L non comparable au ROUGE-Lsum publié |
| 984 000 tokens à 10 % du corpus | l'écart avec le pré-entraîné n'est pas comblable par les données |
| Aucune fuite, sous-ensembles emboîtés | l'ablation mesure la taille, pas la composition |
